# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raju-cse/Flyrank-internship_ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

This notebook audits the Week-5 **Refresh / Content Opportunity Scoring** model.

**Target:** `is_declining_label = (trend_direction == "down")`  
**Week-5 selected model:** Random Forest  
**Primary ranking metric:** Precision@50

The audit focuses on:
1. two constructive methodology questions about the FlyRank research paper;
2. a stricter client-grouped validation of the Week-5 model;
3. leakage checks;
4. real model failures;
5. safer, evidence-matched claims.

All conclusions use public-safe language: **observed, measured, directional, decision-support**.

## 0. Setup — fix the missing-file error

The previous version failed because the notebook was opened in Colab outside the repository root. The Week-5 notebook shows that the correct pipeline creates `data/processed/refresh_feature_vector.csv` from the repository's raw CSV and scripts.

This setup:
- detects whether the notebook is already inside the repository;
- if not, clones the public starter repository into the Colab session;
- installs the repository requirements;
- moves to the repository root;
- runs the feature-preparation and baseline scripts when processed files are missing.

**Do not upload the dataset separately.** The repository's starter data is anonymized and the processed files are regenerated locally.


In [1]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "scripts" / "01_prepare_features.py").exists() and (
            candidate / "data" / "raw" / "content_refresh_anonymized.csv"
        ).exists():
            return candidate
    return None

ROOT = find_repo_root()

if ROOT is None and IN_COLAB:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    ROOT = Path(REPO_DIR).resolve()

if ROOT is None:
    raise FileNotFoundError(
        "Repository root not found. Open this notebook from your FlyRank repository "
        "or run it in Colab so the setup cell can clone the starter repository."
    )

os.chdir(ROOT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)

print("Repository root:", ROOT)
print("Raw data exists:", (ROOT / "data/raw/content_refresh_anonymized.csv").exists())


Repository root: /content/flyrank-ml-internship-starter
Raw data exists: True


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score,
    precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
BASELINE_PATH = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"

if not FEATURE_PATH.exists() or not BASELINE_PATH.exists():
    subprocess.run([sys.executable, "scripts/01_prepare_features.py"], check=True)
    subprocess.run([sys.executable, "scripts/02_baseline_score.py"], check=True)

frame = pd.read_csv(FEATURE_PATH)
baseline_frame = pd.read_csv(BASELINE_PATH)

print("Feature vector:", frame.shape)
print("Baseline:", baseline_frame.shape)


Feature vector: (30000, 52)
Baseline: (30000, 22)


## 1. Two paper findings + my methodology questions

The assignment asks for two findings from the research paper and a constructive methodology question for each.

The questions below are deliberately framed around validation and interpretation rather than grading the paper.

Finding 1 — Content refresh is associated with improved performance
The research reports an observed improvement pattern around refreshed content.

Methodology question: How is the label/outcome for a refreshed page defined, and what is the comparison group? I would want to know whether the analysis compares refreshed pages with an appropriate non-refreshed group over the same time window, and whether the outcome window starts after the refresh. This helps separate an observed before/after pattern from evidence that the refresh itself caused the change.

Finding 2 — Longer content does not reliably explain better search performance
The research tests the common belief that longer content performs better and reports that content length alone is not a reliable explanation.

Methodology question: Does the validation design account for repeated observations from the same client/site and for other factors such as page age, search demand, and ranking position? If related pages or the same client appear on both sides of an evaluation, shared characteristics could make the relationship appear more general than it is.

These are methodology questions, not claims that the findings are incorrect.

## 2. My Week-5 model under an honest split — before / after

The Week-5 notebook used a client-aware split **when the client holdout satisfied the required conditions**, but it also contained a fallback to a stratified row split.

For this audit, I remove that fallback:

- **Before:** the weaker row-level validation is shown as a reference comparison.
- **After:** the same Random Forest setup is evaluated using a strict **client-grouped holdout**.
- No client can appear in both train and test in the audited result.

The repository documentation explicitly describes the reference pipeline as holding out about 20% of clients, not rows.

In [3]:
MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

def build_feature_matrix(df):
    numeric = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
    categorical = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]

    n = (
        df[numeric]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    c = df[categorical].fillna("unknown").astype(str)
    encoded = pd.get_dummies(c, prefix=categorical, dtype=float)

    return pd.concat(
        [n.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1
    )

X = build_feature_matrix(frame)
y = frame["is_declining_label"].astype(int)

print("Model matrix:", X.shape)
print("Target rate:", round(y.mean(), 3))


Model matrix: (30000, 52)
Target rate: 0.542


In [4]:
def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    top = temp.sort_values("score", ascending=False).head(min(k, len(temp)))
    return float(top["y"].mean()) if len(top) else 0.0

def metrics(y_true, scores):
    pred = (np.asarray(scores) >= 0.5).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_true, scores),
        "Average Precision": average_precision_score(y_true, scores),
        "Precision@20": precision_at_k(y_true, scores, 20),
        "Precision@50": precision_at_k(y_true, scores, 50),
        "Precision@100": precision_at_k(y_true, scores, 100),
    }

rf_params = dict(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

# BEFORE: weaker row-level stratified split, included only to show why
# validation design matters.
all_idx = np.arange(len(frame))
before_train, before_test = train_test_split(
    all_idx, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

before_model = RandomForestClassifier(**rf_params)
before_model.fit(X.iloc[before_train], y.iloc[before_train])
before_scores = before_model.predict_proba(X.iloc[before_test])[:, 1]
before_metrics = metrics(y.iloc[before_test], before_scores)

print("BEFORE — stratified row split")
display(pd.DataFrame(before_metrics, index=["value"]).round(3))


BEFORE — stratified row split


,Accuracy,Precision,Recall,F1,ROC AUC,Average Precision,Precision@20,Precision@50,Precision@100
value,0.692,0.709,0.732,0.721,0.758,0.768,0.95,0.9,0.9


In [5]:
# AFTER: strict grouped-by-client split — no row-level fallback.

if "client_id" not in frame.columns:
    raise ValueError("client_id is required for the grouped audit.")

clients = frame["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])

after_test_mask = clients.isin(test_clients).to_numpy()
after_train = np.flatnonzero(~after_test_mask)
after_test = np.flatnonzero(after_test_mask)

if y.iloc[after_train].nunique() != 2 or y.iloc[after_test].nunique() != 2:
    raise ValueError(
        "The strict client holdout does not contain both target classes. "
        "Use a different documented random seed or a larger client sample."
    )

overlap = set(clients.iloc[after_train]) & set(clients.iloc[after_test])
assert len(overlap) == 0

after_model = RandomForestClassifier(**rf_params)
after_model.fit(X.iloc[after_train], y.iloc[after_train])
after_scores = after_model.predict_proba(X.iloc[after_test])[:, 1]
after_metrics = metrics(y.iloc[after_test], after_scores)

print("AFTER — strict client-grouped split")
print("Train rows:", len(after_train))
print("Test rows:", len(after_test))
print("Train clients:", clients.iloc[after_train].nunique())
print("Test clients:", clients.iloc[after_test].nunique())
print("Client overlap:", len(overlap))
display(pd.DataFrame(after_metrics, index=["value"]).round(3))


AFTER — strict client-grouped split
Train rows: 27675
Test rows: 2325
Train clients: 26
Test clients: 6
Client overlap: 0


,Accuracy,Precision,Recall,F1,ROC AUC,Average Precision,Precision@20,Precision@50,Precision@100
value,0.672,0.561,0.744,0.64,0.75,0.618,0.65,0.74,0.72


In [6]:
comparison = pd.DataFrame({
    "Before — row split": before_metrics,
    "After — client grouped": after_metrics,
})

comparison["Change"] = (
    comparison["After — client grouped"] -
    comparison["Before — row split"]
)

display(comparison.round(3))


,Before — row split,After — client grouped,Change
Accuracy,0.692,0.672,-0.020
Precision,0.709,0.561,-0.148
Recall,0.732,0.744,0.011
F1,0.721,0.640,-0.081
ROC AUC,0.758,0.750,-0.008
Average Precision,0.768,0.618,-0.150
Precision@20,0.950,0.650,-0.300
Precision@50,0.900,0.740,-0.160
Precision@100,0.900,0.720,-0.180


Interpretation
The after result is the one to use for the validation claim because it evaluates unseen clients.

A drop from the row-level result to the grouped result is not a failure of the notebook. It is useful evidence that row-level validation can be optimistic when related pages share client characteristics.

If the grouped result remains similar, that is evidence of greater stability under this particular split.

In either case, the result is measured on this dataset and split. It does not establish universal performance or causation.

Week-5 reference result
The uploaded Week-5 notebook reports the reference pipeline's approximate Random Forest result as:

Precision@50 ≈ 0.74
ROC-AUC ≈ 0.750
The repository itself notes that the live model number can vary with library versions. The executed tables above are authoritative for the current environment.

This audit therefore avoids treating the old value as a universal benchmark.

## 3. Leakage audit

The target is explicitly derived from:

`is_declining_label = (trend_direction == "down")`

Therefore:

- `trend_direction` must not be a model feature.
- `trend_pct` must not be a model feature because it directly describes the outcome trend.
- `client_id` is used only to create the grouped split.
- `content_id` is an identifier and is not used as a predictive feature.

The Week-5 feature lists are reused here so the audit evaluates the same model concept rather than silently introducing unrelated features.


In [7]:
# Explicit feature-level leakage audit.

audit_rows = [
    {
        "feature": "trend_direction",
        "risk": "HIGH",
        "reason": "Directly defines the target label.",
        "action": "Exclude"
    },
    {
        "feature": "trend_pct",
        "risk": "HIGH",
        "reason": "Outcome-trend information can expose the target.",
        "action": "Exclude"
    },
    {
        "feature": "client_id",
        "risk": "HIGH",
        "reason": "Identifier; may enable memorization rather than generalization.",
        "action": "Use only for grouped splitting"
    },
    {
        "feature": "content_id",
        "risk": "HIGH",
        "reason": "Identifier rather than a predictive measurement.",
        "action": "Exclude"
    },
]

for forbidden in ["trend_direction", "trend_pct", "client_id", "content_id"]:
    assert forbidden not in MODEL_NUMERIC_FEATURES
    assert forbidden not in MODEL_CATEGORICAL_FEATURES

leakage_audit = pd.DataFrame(audit_rows)
display(leakage_audit)

print("Leakage guard passed for direct target and identifier columns.")


,feature,risk,reason,action
0,trend_direction,HIGH,Directly defines the target label.,Exclude
1,trend_pct,HIGH,Outcome-trend information can expose the target.,Exclude
2,client_id,HIGH,Identifier; may enable memorization rather tha...,Use only for grouped splitting
3,content_id,HIGH,Identifier rather than a predictive measurement.,Exclude


Leakage guard passed for direct target and identifier columns.


### Additional timing question

The current feature vector contains historical 90-day measurements and content-age/freshness variables. For a production decision, the next audit question would be whether **every feature was available before the prediction cutoff**.

That temporal availability question cannot be completely proven from the executed model notebook alone. It should be verified against the feature-generation timestamps/data contract before deployment.

This is intentionally stated as an open methodology check rather than an unsupported assurance.


## 4. Real failure examples

The examples below use only the anonymized identifiers and model-facing measurements already present in the starter dataset.

A false positive means the model ranked a page as likely declining but the observed label is not declining. A false negative means the model ranked a declining page lower than the 0.5 classification threshold.


In [8]:
error_frame = frame.iloc[after_test][[
    c for c in [
        "content_id", "client_id", "is_declining_label", "trend_direction",
        "impressions_90d", "sessions_90d", "avg_position", "ctr",
        "word_count", "days_since_last_update"
    ] if c in frame.columns
]].copy()

error_frame["predicted_probability"] = after_scores

error_frame["error_type"] = np.where(
    (error_frame["is_declining_label"] == 0) &
    (error_frame["predicted_probability"] >= 0.5),
    "false_positive",
    np.where(
        (error_frame["is_declining_label"] == 1) &
        (error_frame["predicted_probability"] < 0.5),
        "false_negative",
        "correct"
    )
)

print("Error counts")
display(error_frame["error_type"].value_counts().to_frame("count"))

print("Highest-confidence false positives")
display(
    error_frame[error_frame["error_type"] == "false_positive"]
    .sort_values("predicted_probability", ascending=False)
    .head(10)
)

print("Highest-confidence false negatives")
display(
    error_frame[error_frame["error_type"] == "false_negative"]
    .sort_values("predicted_probability", ascending=True)
    .head(10)
)


Error counts


,count
error_type,
correct,1563
false_positive,529
false_negative,233


Highest-confidence false positives


,content_id,client_id,is_declining_label,trend_direction,impressions_90d,sessions_90d,avg_position,ctr,word_count,days_since_last_update,predicted_probability,error_type
23250,content_d2dffcc697a4,client_f74efabef1,0,stable,5091,30,14.1,0.20,4496.0,20,0.737130,false_positive
23559,content_00603b0349b4,client_f74efabef1,0,up,1076,9,25.6,0.09,2439.0,20,0.734944,false_positive
25913,content_331182ca4cae,client_f74efabef1,0,up,3026,24,35.9,0.00,3546.0,20,0.733631,false_positive
23750,content_e55b8ab078b0,client_f74efabef1,0,stable,369,2,21.8,0.00,2192.0,20,0.733059,false_positive
10155,content_643f585dc7f7,client_f74efabef1,0,up,761,6,25.1,0.39,1980.0,20,0.731120,false_positive
5966,content_f5013794ba57,client_f74efabef1,0,new,881,2,15.7,0.00,3622.0,20,0.730532,false_positive
28337,content_ea4417d89e2c,client_f74efabef1,0,stable,352,4,11.9,0.00,2556.0,20,0.729056,false_positive
4249,content_db1cd41b4b4f,client_f74efabef1,0,up,1482,19,12.9,0.00,2221.0,105,0.729052,false_positive
21530,content_b15a8dbdf66f,client_f74efabef1,0,up,1647,10,22.4,0.18,4095.0,20,0.727853,false_positive
2380,content_96da95476e63,client_f74efabef1,0,stable,784,6,7.4,0.00,2598.0,20,0.724807,false_positive


Highest-confidence false negatives


,content_id,client_id,is_declining_label,trend_direction,impressions_90d,sessions_90d,avg_position,ctr,word_count,days_since_last_update,predicted_probability,error_type
5770,content_28b4223f4e5f,client_98a3ab7c34,1,down,1,1,0.0,0.00,3109.0,1,0.079867,false_negative
3879,content_34b14c00f80c,client_d4735e3a26,1,down,3,1,0.0,0.00,659.0,20,0.082196,false_negative
27177,content_79ac977c6e0b,client_f74efabef1,1,down,3,1,0.7,0.00,2304.0,8,0.149546,false_negative
22991,content_472ce7ae14c0,client_d4735e3a26,1,down,3,2,0.3,33.33,684.0,20,0.152184,false_negative
5608,content_a55d958ec725,client_d4735e3a26,1,down,3,1,2.7,0.00,837.0,20,0.163987,false_negative
12864,content_f1ef151d5e36,client_d4735e3a26,1,down,3,2,2.0,0.00,978.0,20,0.165946,false_negative
12076,content_230de4c50860,client_d4735e3a26,1,down,3,1,2.0,0.00,824.0,20,0.169816,false_negative
25838,content_cbc3b52a2ac1,client_98a3ab7c34,1,down,2,1,3.0,0.00,2286.0,1,0.171345,false_negative
13659,content_4c437dd8c1ee,client_d4735e3a26,1,down,3,1,3.0,0.00,840.0,20,0.174066,false_negative
23810,content_37804210415c,client_d4735e3a26,1,down,4,3,2.0,0.00,813.0,20,0.174828,false_negative


### What the failures tell me

The failures show where a ranked review model can disagree with the observed label. They are useful for deciding where human review or additional features may be valuable.

They do **not** show that a single feature causes decline. The appropriate interpretation is directional and decision-support oriented.


## 5. Claim rewrite

### Claim that goes too far

> “Our Random Forest accurately predicts which pages will decline and is about 3× better than the baseline.”

### Evidence-matched rewrite

> **Observed:** The Random Forest produced a measured improvement in the evaluated ranking metrics relative to the Week-4 rule on the tested holdout. Under the stricter client-grouped audit, the model's performance was re-measured on unseen clients. **Directionally**, the scores can support prioritizing pages for manual review. This is **decision-support** evidence for the evaluated dataset and split, not proof of universal predictive performance or causation.

### Another claim to avoid

> “The model predicts Google's algorithm.”

### Safer wording

> The model ranks anonymized content pages for a defined declining-content classification task using the available features.


In [9]:
claim_check = pd.DataFrame([
    {
        "statement": "Random Forest performance",
        "status": "Measured",
        "safe_language": "observed / measured"
    },
    {
        "statement": "Use model ranking to prioritize pages for review",
        "status": "Directional decision support",
        "safe_language": "directional / decision-support"
    },
    {
        "statement": "Model predicts Google's algorithm",
        "status": "Not supported",
        "safe_language": "Do not claim"
    },
    {
        "statement": "A feature causes content decline",
        "status": "Not established by this notebook",
        "safe_language": "Do not claim causation"
    },
])

display(claim_check)


,statement,status,safe_language
0,Random Forest performance,Measured,observed / measured
1,Use model ranking to prioritize pages for review,Directional decision support,directional / decision-support
2,Model predicts Google's algorithm,Not supported,Do not claim
3,A feature causes content decline,Not established by this notebook,Do not claim causation


## 6. Self-check

- [x] Two paper findings are named and paired with constructive methodology questions.
- [x] Week-5 Random Forest setup is reused.
- [x] A before/after validation comparison is included.
- [x] The after evaluation uses a strict client-grouped holdout with no row-level fallback.
- [x] Client overlap is asserted to be zero.
- [x] Direct target leakage is audited.
- [x] Real anonymized false-positive and false-negative examples are shown.
- [x] Claims are rewritten using observed / measured / directional / decision-support language.
- [ ] Run **Runtime → Run all** from the repository root and keep the outputs.
- [ ] Save/commit as `work/notebooks/w06_validation_audit.ipynb`.
- [ ] Submit the GitHub repository URL on the FlyRank card.

### Final conclusion

The audit supports a narrow conclusion: the Random Forest can be **measured as a ranking/decision-support model for the defined declining-content task** on the evaluated anonymized dataset. The grouped validation gives a more conservative test of cross-client generalization. Broader claims should wait for further temporal and out-of-sample validation.
